In [3]:
import pandas as pd
import random

reasons = [
    "Too Expensive",
    "Not Useful",
    "Found Alternative",
    "Technical Issues",
    "Lack of Content"
]

data = []

for i in range(1000):
    data.append({
        "user_id": i,
        "subscription_months": random.randint(1,12),
        "cancel_reason": random.choice(reasons),
        "last_active_days": random.randint(1,60),
        "plan_type": random.choice(["Basic","Premium"]),
        "churn": random.choice([0,1])
    })

df = pd.DataFrame(data)

df.to_csv("users.csv",index=False)

import pandas as pd
import random

df = pd.read_csv("users.csv")

df["login_frequency"] = [
    random.randint(0,5) if churn == 1
    else random.randint(10,30)
    for churn in df["churn"]
]

df.to_csv("users.csv",index=False)

In [4]:
import sqlite3
import pandas as pd

df = pd.read_csv("users.csv")

conn = sqlite3.connect("retention.db")

df.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

print("Database created")

Database created


In [5]:
query1 = """
SELECT cancel_reason,
COUNT(*) AS users
FROM users
WHERE churn = 1
GROUP BY cancel_reason
ORDER BY users DESC;
"""

print("\n TOP CHURN REASONS ")
top_reasons_df = pd.read_sql_query(query1, conn)

print(top_reasons_df)


 TOP CHURN REASONS 
       cancel_reason  users
0         Not Useful    103
1  Found Alternative    103
2   Technical Issues     95
3    Lack of Content     92
4      Too Expensive     86


In [6]:
query2 = """
SELECT plan_type,
ROUND(AVG(churn) * 100, 2) AS churn_rate
FROM users
GROUP BY plan_type;
"""

print("\n CHURN RATE BY PLAN ")
plan_churn_df = pd.read_sql_query(query2, conn)

print(plan_churn_df)


 CHURN RATE BY PLAN 
  plan_type  churn_rate
0     Basic       50.71
1   Premium       45.19


In [7]:
query3 = """
SELECT ROUND(AVG(subscription_months),2)
AS avg_subscription_months
FROM users;
"""

print("\n AVERAGE SUBSCRIPTION MONTHS")
print(pd.read_sql_query(query3, conn))




 AVERAGE SUBSCRIPTION MONTHS
   avg_subscription_months
0                     6.46


In [8]:
query4 = """
SELECT ROUND(AVG(subscription_months),2)
AS avg_duration
FROM users
WHERE churn = 1;
"""

print("\n AVG SUBSCRIPTION OF CHURNED USERS ")
print(pd.read_sql_query(query4, conn))



 AVG SUBSCRIPTION OF CHURNED USERS 
   avg_duration
0          6.51


In [9]:
query5 = """
SELECT
CASE
    WHEN churn = 1 THEN 'Churned'
    ELSE 'Active'
END AS status,
COUNT(*) AS users
FROM users
GROUP BY churn;
"""

print("\n ACTIVE VS CHURNED USERS ")
print(pd.read_sql_query(query5, conn))



 ACTIVE VS CHURNED USERS 
    status  users
0   Active    521
1  Churned    479


In [10]:
query6 = """
SELECT subscription_months,
COUNT(*) AS churn_count
FROM users
WHERE churn = 1
GROUP BY subscription_months
ORDER BY subscription_months;
"""

print("\n CHURN BY SUBSCRIPTION MONTH ")
print(pd.read_sql_query(query6, conn))



 CHURN BY SUBSCRIPTION MONTH 
    subscription_months  churn_count
0                     1           34
1                     2           35
2                     3           46
3                     4           48
4                     5           38
5                     6           39
6                     7           43
7                     8           33
8                     9           45
9                    10           42
10                   11           43
11                   12           33


In [11]:
query7 = """
SELECT churn,
ROUND(AVG(last_active_days),2)
AS avg_last_active
FROM users
GROUP BY churn;
"""

print("\n LAST ACTIVE DAYS ANALYSIS ")
print(pd.read_sql_query(query7, conn))



 LAST ACTIVE DAYS ANALYSIS 
   churn  avg_last_active
0      0            31.53
1      1            30.18


In [12]:
query8 = """
SELECT plan_type,
COUNT(*) AS users
FROM users
GROUP BY plan_type;
"""

print("\n PLAN DISTRIBUTION ")
print(pd.read_sql_query(query8, conn))



 PLAN DISTRIBUTION 
  plan_type  users
0     Basic    491
1   Premium    509


In [13]:
query9 = """
SELECT
cancel_reason,
ROUND(
COUNT(*) * 100.0 /
(SELECT COUNT(*) FROM users WHERE churn=1),
2
) AS percentage
FROM users
WHERE churn=1
GROUP BY cancel_reason
ORDER BY percentage DESC;
"""

print("\n CHURN PERCENTAGE BY REASON ")
print(pd.read_sql_query(query9, conn))



 CHURN PERCENTAGE BY REASON 
       cancel_reason  percentage
0         Not Useful       21.50
1  Found Alternative       21.50
2   Technical Issues       19.83
3    Lack of Content       19.21
4      Too Expensive       17.95


In [14]:
query10 = """
SELECT
COUNT(*) AS total_users,
SUM(churn) AS churned_users,
ROUND(AVG(subscription_months),2)
AS avg_subscription
FROM users;
"""

print("\n EXECUTIVE SUMMARY ")
summary_df = pd.read_sql_query(query10, conn)

print(summary_df)
print(top_reasons_df)
conn.close()



 EXECUTIVE SUMMARY 
   total_users  churned_users  avg_subscription
0         1000            479              6.46
       cancel_reason  users
0         Not Useful    103
1  Found Alternative    103
2   Technical Issues     95
3    Lack of Content     92
4      Too Expensive     86


In [15]:
print("\n===== ANALYSIS COMPLETE =====")
print("SQL-based churn analysis executed successfully.")
print("Data ready for AI insight generation.")


===== ANALYSIS COMPLETE =====
SQL-based churn analysis executed successfully.
Data ready for AI insight generation.


In [16]:
print(top_reasons_df.head())
print(plan_churn_df.head())
print(summary_df.head())

       cancel_reason  users
0         Not Useful    103
1  Found Alternative    103
2   Technical Issues     95
3    Lack of Content     92
4      Too Expensive     86
  plan_type  churn_rate
0     Basic       50.71
1   Premium       45.19
   total_users  churned_users  avg_subscription
0         1000            479              6.46


In [17]:
from google import genai

client = genai.Client(
    api_key="AQ.Ab8RN6JWqaoOaREuZRWQVx6nnycOKrJ9Q3ID4fA-P-mQHvFZoA"
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="""
    Analyze this churn data and provide
    retention recommendations.
    """
)

print(response.text)

Okay, I can definitely help you structure an analysis of churn data and provide robust retention recommendations.

Since you haven't provided the actual churn data, I'll outline the **process I would follow to analyze the data** and then give you **comprehensive, actionable retention recommendations** that you can tailor based on the insights from your specific dataset.

---

## Part 1: How I Would Analyze Your Churn Data (Hypothetical Analysis Framework)

To effectively analyze your churn data, I would look for patterns, segments, and root causes. Here's a breakdown of the steps and dimensions I'd investigate:

### 1. Key Churn Metrics & Trends:

*   **Overall Churn Rate:**
    *   **Customer Churn Rate:** (Number of churned customers / Total customers at start of period) * 100%
    *   **Revenue Churn Rate (Gross):** (Lost MRR from churn / Total MRR at start of period) * 100%
    *   **Net Revenue Retention (NRR):** (Starting MRR + Expansion MRR - Contraction MRR - Churn MRR / Starti

In [19]:
summary = f"""
TOP CHURN REASONS

{top_reasons_df.to_string(index=False)}

PLAN WISE CHURN

{plan_churn_df.to_string(index=False)}

EXECUTIVE SUMMARY

{summary_df.to_string(index=False)}
"""

In [20]:
prompt = f"""
You are a Senior Product and Retention Strategist.

Analyze the following churn metrics.

{summary}

Provide:

1. Executive Summary

2. Key Churn Drivers

3. User Behaviour Patterns

4. Retention Strategies

5. Product Recommendations

6. Business Impact

Keep recommendations specific, actionable and data-driven.
"""

In [21]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

## Churn Metrics Analysis: Strategic Review

**Overall Assessment:** The current churn rate of 47.9% is critically high and unsustainable for long-term growth. Nearly half of the user base is churning, with the average subscription duration of 6.46 months indicating that users are giving the product a chance but ultimately finding it insufficient or frustrating. The churn reasons are diverse, suggesting a multi-faceted problem that requires a comprehensive strategy across product, operations, and marketing.

---

### 1. Executive Summary

Our platform is experiencing an alarmingly high churn rate of **47.9%**, with 479 out of 1000 total users churning. Users subscribe for an average of 6.46 months before deciding to leave, indicating a delayed but significant dissatisfaction. The primary drivers for churn are "Not Useful" and "Found Alternative" (both 103 users), closely followed by "Technical Issues" (95 users), "Lack of Content" (92 users), and "Too Expensive" (86 users). While Basic

In [22]:
with open(
    "retention_report.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(response.text)

print("\nAI Report Generated Successfully")


AI Report Generated Successfully
